# Maintainer's Copilot — DistilBERT Classifier (Colab T4)

Fine-tunes `distilbert-base-uncased` for 4-class GitHub issue triage:  
`bug` / `feature` / `docs` / `question`

## Before running

1. **Runtime → Change runtime type → T4 GPU**
2. Run `python backend/scripts/download_splits.py` locally to get the split files
3. Upload these three files to Google Drive at exactly these paths:
   - `MyDrive/maintainers-copilot/splits/train.jsonl`
   - `MyDrive/maintainers-copilot/splits/val.jsonl`
   - `MyDrive/maintainers-copilot/splits/test.jsonl`
4. Run all cells top to bottom

## After training

Weights are saved to `MyDrive/maintainers-copilot/models/weights.pt`.  
Download that file locally and run `python backend/scripts/upload_weights.py` to push it to MinIO.

In [ ]:
# Cell 2 — Install dependencies
!pip install --quiet transformers torch scikit-learn

In [ ]:
# Cell 3 — Mount Google Drive and set paths
import uuid
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE   = Path("/content/drive/MyDrive/maintainers-copilot")
SPLITS_DIR   = DRIVE_BASE / "splits"
MODELS_DIR   = DRIVE_BASE / "models"
CKPT_DIR     = MODELS_DIR / "checkpoints"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = str(uuid.uuid4())
print(f"Run ID    : {RUN_ID}")
print(f"Splits dir: {SPLITS_DIR}")
print(f"Models dir: {MODELS_DIR}")

In [ ]:
# Cell 4 — Verify split files exist in Drive
for split in ("train", "val", "test"):
    path = SPLITS_DIR / f"{split}.jsonl"
    assert path.exists(), (
        f"Missing: {path}\n"
        "Upload the split files to MyDrive/maintainers-copilot/splits/ first."
    )
    lines = path.read_text().strip().splitlines()
    print(f"{split:<6}: {len(lines)} rows  ({path})")

In [ ]:
# Cell 5 — Load splits from Drive
import json

def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

train_data = load_jsonl(SPLITS_DIR / "train.jsonl")
val_data   = load_jsonl(SPLITS_DIR / "val.jsonl")

print(f"Train rows : {len(train_data)}")
print(f"Val rows   : {len(val_data)}")

In [ ]:
# Cell 6 — Class distribution and first 3 examples
from collections import Counter

LABEL2ID = {"bug": 0, "feature": 1, "docs": 2, "question": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def show_distribution(name: str, data: list[dict]) -> None:
    counts = Counter(row["label"] for row in data)
    total = len(data)
    print(f"\n{name} ({total} rows):")
    for cls in ("bug", "feature", "docs", "question"):
        n = counts.get(cls, 0)
        print(f"  {cls:<10} {n:>5}  ({100*n/total:.1f}%)")

show_distribution("train", train_data)
show_distribution("val",   val_data)

print("\n--- First 3 training examples ---")
for row in train_data[:3]:
    preview = row["text"][:120].replace("\n", " ")
    print(f"[{row['label']}] {preview} …")

In [ ]:
# Cell 7 — Tokenizer setup
from transformers import DistilBertTokenizerFast

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

sample = tokenizer(["Test issue title"], truncation=True, padding="max_length",
                   max_length=MAX_LENGTH, return_tensors="pt")
print(f"Tokenizer OK. Input IDs shape: {sample['input_ids'].shape}")

In [ ]:
# Cell 8 — IssueDataset (torch Dataset)
import torch
from torch.utils.data import Dataset

class IssueDataset(Dataset):
    """Tokenised GitHub issues dataset for DistilBERT fine-tuning."""

    def __init__(self, rows: list[dict], label2id: dict[str, int]) -> None:
        texts  = [row["text"] for row in rows]
        labels = [label2id[row["label"]] for row in rows]
        enc = tokenizer(texts, truncation=True, padding="max_length",
                        max_length=MAX_LENGTH, return_tensors="pt")
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

print("Building datasets (this may take a minute) …")
train_dataset = IssueDataset(train_data, LABEL2ID)
val_dataset   = IssueDataset(val_data,   LABEL2ID)
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset  : {len(val_dataset)} samples")

In [ ]:
# Cell 9 — Model setup
from transformers import DistilBertForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model = model.to(DEVICE)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 10 — Freeze all but last transformer block + classifier head
for param in model.parameters():
    param.requires_grad = False

for param in model.distilbert.transformer.layer[5].parameters():
    param.requires_grad = True
for param in model.pre_classifier.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable : {trainable:,}")
print(f"Frozen    : {frozen:,}")
print(f"Fraction  : {trainable / (trainable + frozen):.1%}")

In [ ]:
# Cell 11 — Training loop (checkpoints saved to Google Drive)
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

BATCH_SIZE   = 32
NUM_EPOCHS   = 5
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY,
)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps,
)

print(f"Steps/epoch: {len(train_loader)}  Total: {total_steps}  Warmup: {WARMUP_STEPS}")

def save_checkpoint(epoch: int) -> None:
    path = CKPT_DIR / f"checkpoint_epoch_{epoch}_{RUN_ID}.pt"
    torch.save(model.state_dict(), path)
    print(f"  Checkpoint saved → {path}")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += outputs.loss.item()
        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch} step {step+1}/{len(train_loader)}  "
                  f"loss={total_loss/(step+1):.4f}")

    print(f"Epoch {epoch}/{NUM_EPOCHS} — avg loss: {total_loss/len(train_loader):.4f}")
    save_checkpoint(epoch)

In [ ]:
# Cell 12 — Evaluate on validation set
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

def evaluate(loader: DataLoader, split_name: str) -> dict:
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().tolist())
            all_labels.extend(batch["labels"].tolist())

    label_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]
    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    print(f"\n{'='*50}\n{split_name}\n{'='*50}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro-F1 : {f1_macro:.4f}")
    print(classification_report(all_labels, all_preds, target_names=label_names))
    print(confusion_matrix(all_labels, all_preds))
    return {"accuracy": acc, "f1_macro": f1_macro}

val_metrics = evaluate(val_loader, "Validation")

In [ ]:
# Cell 13 — Final evaluation on test set
test_data    = load_jsonl(SPLITS_DIR / "test.jsonl")
test_dataset = IssueDataset(test_data, LABEL2ID)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_metrics = evaluate(test_loader, "Test (held-out)")

CI_ACC = 0.70
CI_F1  = 0.65
print(f"\nCI gate: accuracy >= {CI_ACC}: {'PASS' if test_metrics['accuracy'] >= CI_ACC else 'FAIL'}")
print(f"CI gate: f1_macro >= {CI_F1}:  {'PASS' if test_metrics['f1_macro']  >= CI_F1  else 'FAIL'}")

In [ ]:
# Cell 14 — Save weights to Google Drive, compute SHA-256, print run summary
import hashlib

weights_path = MODELS_DIR / "weights.pt"
torch.save(model.state_dict(), weights_path)

weights_bytes  = weights_path.read_bytes()
weights_sha256 = hashlib.sha256(weights_bytes).hexdigest()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Run ID         : {RUN_ID}")
print(f"Weights SHA-256: {weights_sha256}")
print(f"Weights path   : {weights_path}")
print(f"Val  accuracy  : {val_metrics['accuracy']:.4f}")
print(f"Val  macro-F1  : {val_metrics['f1_macro']:.4f}")
print(f"Test accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Test macro-F1  : {test_metrics['f1_macro']:.4f}")
print("="*60)
print("\nNext: download weights.pt from Google Drive, then run:")
print("  python backend/scripts/upload_weights.py")

## Next steps

1. Copy **Run ID** and **SHA-256** from Cell 14 into `model_card.md`.
2. Download `weights.pt` from `MyDrive/maintainers-copilot/models/weights.pt`.
3. Place it at `backend/weights/weights.pt` locally.
4. Run `python backend/scripts/upload_weights.py` to push it to MinIO.
5. Proceed to **Phase 5** — classical ML + LLM baselines.